# SynthTextEval evaluation pipeline for ICL-PILOT

Evaluate the three pinned generators separately with common real reference rows and age-stratified, child/prompt-pair-grouped classifier folds using all eligible ENNI children. Set the explicit ENNI/generated and Gillam CSV paths in `configs/evaluation/inputs.json`.

Synthetic training data are bootstrapped within age and clinical group to preserve each real training fold's proportions. Text fidelity, interpretable linguistic fidelity and classification utility are distinct outcomes. Gillam remains external testing only. See `docs/guides/framework_evaluation.md`.


In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module=r"jupyter_client.*",
)


In [ ]:
# Repository paths and run configuration.
from pathlib import Path
import sys

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
COLAB_REPO_DIR = Path("/content/ICL-PILOT")
candidate_roots = [COLAB_REPO_DIR, Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in candidate_roots if (path / "pyproject.toml").exists()), None)

if PROJECT_ROOT is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Run this notebook from the ICL-PILOT repository or open it in Colab.")
    !git clone {REPO_URL} {COLAB_REPO_DIR}
    PROJECT_ROOT = COLAB_REPO_DIR

%cd {PROJECT_ROOT}
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from icl_pilot.evaluation.runs import create_framework_evaluation_run
from icl_pilot.evaluation.inputs import load_evaluation_inputs

EVALUATION_INPUT_CONFIG = PROJECT_ROOT / "configs/evaluation/inputs.json"
EVALUATION_RUN_ID = None  # None creates a UTC timestamp; an explicit name must be unused.
PROTOCOL_CONFIG = PROJECT_ROOT / "configs/evaluation/protocol.json"
RUN_CLASSIFICATION = True  # Set False for input/descriptive checks without BERT fitting.
RUN_TEXT_QUALITY = True  # Downloads pinned GPT-2 and sentence-encoder checkpoints.

# Export settings. Model checkpoints are large and are excluded from the portable
# ZIP by default; predictions, metrics, split definitions, manifests, and plots are kept.
DRIVE_EXPORT_ROOT = Path("/content/drive/MyDrive/ICL-PILOT/framework_based_evaluation")
EXPORT_INCLUDE_MODEL_CHECKPOINTS = False
EXPORT_ARCHIVE_PREFIX = "framework_based_evaluation"

evaluation_inputs = load_evaluation_inputs(PROJECT_ROOT, config_path=EVALUATION_INPUT_CONFIG)
TEXT_RECORDS_CSV = evaluation_inputs["text_records_csv"]
GILLAM_EXTERNAL_TEXT_CSV = evaluation_inputs["gillam_external_text_csv"]
RESULTS_DIR = create_framework_evaluation_run(PROJECT_ROOT, run_id=EVALUATION_RUN_ID)
print({
    "project_root": str(PROJECT_ROOT),
    "text_records": str(TEXT_RECORDS_CSV),
    "gillam_external_records": str(GILLAM_EXTERNAL_TEXT_CSV),
    "results_dir": str(RESULTS_DIR),
    "evaluation_run_id": RESULTS_DIR.name,
})


In [ ]:
# Install SynthTextEval from a local clone and apply narrowly scoped compatibility fixes.
# A sentinel prevents a reinstall loop after the one required runtime restart.
from pathlib import Path
import re
import subprocess
import sys

SYNTHTEXTEVAL_REVISION = "3d63a172418a985fd0d997e5c993f6a0390c294e"
SYNTHTEXTEVAL_DIR = Path("/content/synthtexteval-src")
INSTALL_SENTINEL = SYNTHTEXTEVAL_DIR / ".icl_pilot_install_ready"


def run_checked(command):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(part) for part in command], check=True)


if not (SYNTHTEXTEVAL_DIR / ".git").exists():
    run_checked(["git", "clone", "https://github.com/kr-ramesh/synthtexteval.git", SYNTHTEXTEVAL_DIR])

sentinel_matches = (
    INSTALL_SENTINEL.exists()
    and INSTALL_SENTINEL.read_text(encoding="utf-8").strip() == SYNTHTEXTEVAL_REVISION
)

if not sentinel_matches:
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "fetch", "--all", "--tags", "-q"])
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "checkout", "-q", SYNTHTEXTEVAL_REVISION])
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "reset", "--hard", "-q", SYNTHTEXTEVAL_REVISION])

    # Packaging fix: this repository revision lacks synthtexteval/eval/__init__.py,
    # while setup.py relies on package discovery.
    eval_init = SYNTHTEXTEVAL_DIR / "synthtexteval" / "eval" / "__init__.py"
    eval_init.touch(exist_ok=True)

    # Transformers compatibility fix. Keep tokenization semantics unchanged:
    # encode_plus -> tokenizer(...) and pad_to_max_length=True -> padding="max_length".
    classifier_utils_path = (
        SYNTHTEXTEVAL_DIR
        / "synthtexteval"
        / "eval"
        / "downstream"
        / "classify"
        / "utils.py"
    )
    utils_text = classifier_utils_path.read_text(encoding="utf-8")
    utils_text, encode_plus_count = re.subn(
        r"tokenizer\s*\.\s*encode_plus\s*\(",
        "tokenizer(",
        utils_text,
    )
    utils_text, legacy_padding_count = re.subn(
        r"pad_to_max_length\s*=\s*True",
        'padding="max_length"',
        utils_text,
    )
    classifier_utils_path.write_text(utils_text, encoding="utf-8")

    assert re.search(r"tokenizer\s*\.\s*encode_plus\s*\(", utils_text) is None
    assert "pad_to_max_length" not in utils_text
    assert 'padding="max_length"' in utils_text

    print({
        "synthtexteval_revision": SYNTHTEXTEVAL_REVISION,
        "encode_plus_replacements": encode_plus_count,
        "legacy_padding_replacements": legacy_padding_count,
        "source": str(SYNTHTEXTEVAL_DIR),
    })

    # Install repository requirements first, then pin Transformers last so an unpinned
    # transformers requirement cannot leave a v5 installation active.
    run_checked([sys.executable, "-m", "pip", "uninstall", "-y", "SynthTextEval"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "-r", SYNTHTEXTEVAL_DIR / "requirements.txt"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "accelerate>=0.26.0"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", SYNTHTEXTEVAL_DIR])

    INSTALL_SENTINEL.write_text(SYNTHTEXTEVAL_REVISION + "\n", encoding="utf-8")
    SYNTH_INSTALL_RESTART_REQUIRED = True
    print("\nSynthTextEval installation/patch completed.")
    print("Restart the runtime ONCE now, then rerun the notebook from the top.")
else:
    SYNTH_INSTALL_RESTART_REQUIRED = False
    print("Pinned SynthTextEval installation already prepared in this runtime; reinstall skipped.")


In [ ]:
# Run after restarting the kernel/runtime. Verify that Python is using the patched clone.
from pathlib import Path
import inspect
import synthtexteval
import transformers

from synthtexteval.eval.downstream.classify import utils as classifier_utils
from synthtexteval.eval.text_quality import metrics as text_quality_metrics

loaded_utils_path = Path(classifier_utils.__file__).resolve()
expected_source_root = Path("/content/synthtexteval-src").resolve()

print("SynthTextEval:", synthtexteval.__file__)
print("Transformers:", transformers.__version__)
print("Classifier utils:", loaded_utils_path)

assert expected_source_root in loaded_utils_path.parents, (
    f"Wrong SynthTextEval copy loaded: {loaded_utils_path}. Expected the editable clone under {expected_source_root}."
)
assert transformers.__version__ == "4.57.6", "Restart after installing the pinned framework runtime."

tokenize_source = inspect.getsource(classifier_utils.tokenize_data)
assert "encode_plus" not in tokenize_source, "Old encode_plus call is still active."
assert "pad_to_max_length" not in tokenize_source, "Old pad_to_max_length argument is still active."
assert "padding" in tokenize_source and "max_length" in tokenize_source

# The public README uses QualEval, but tolerate an older class name if a repository
# revision exposes it instead. The text-quality cell uses the same resolution logic.
quality_class = getattr(text_quality_metrics, "QualEval", None)
if quality_class is None:
    quality_class = getattr(text_quality_metrics, "TextQualityEval", None)
assert quality_class is not None, (
    "No supported text-quality evaluator class (QualEval/TextQualityEval) was found in "
    f"{text_quality_metrics.__file__}."
)
print("Text-quality evaluator:", quality_class.__name__)
print("SynthTextEval installation/import checks passed.")

import subprocess
assert subprocess.check_output(["git", "-C", str(SYNTHTEXTEVAL_DIR), "rev-parse", "HEAD"], text=True).strip() == SYNTHTEXTEVAL_REVISION, "Unexpected SynthTextEval source revision."


In [ ]:
import hashlib
import json
import pandas as pd
from icl_pilot.evaluation.framework_data import load_protocol, prepare_evaluation

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

protocol = load_protocol(PROJECT_ROOT, PROTOCOL_CONFIG)
records = pd.read_csv(TEXT_RECORDS_CSV, dtype={"source_child_id": str, "td_child_id": str, "sli_child_id": str})
gillam = pd.read_csv(GILLAM_EXTERNAL_TEXT_CSV, dtype={"source_child_id": str})
# Preserve the earlier explicitly identified meta-response criterion. Do not
# silently change the three generators' matched schedule through exclusions.
meta = records["source"].eq("synthetic") & records["story_text"].str.contains(
    "I replaced the main characters to match the target story", case=False, regex=False, na=False)
records.loc[meta].to_csv(RESULTS_DIR / "flagged_meta_generations.csv", index=False)
if meta.any():
    raise ValueError("Flagged meta responses are preserved. Resolve the common schedule/QC rule before scientific fitting.")
prepared = prepare_evaluation(records, gillam, protocol, RESULTS_DIR)
(RESULTS_DIR / "input_file_hashes.json").write_text(json.dumps({
    str(TEXT_RECORDS_CSV): sha256_file(TEXT_RECORDS_CSV),
    str(GILLAM_EXTERNAL_TEXT_CSV): sha256_file(GILLAM_EXTERNAL_TEXT_CSV),
    str(PROTOCOL_CONFIG): sha256_file(PROTOCOL_CONFIG),
}, indent=2) + "\n")
display(pd.read_csv(RESULTS_DIR / "split_audit.csv"))


In [ ]:
# The same real reference sample is reused across generators, balanced within
# age cohort, literal group and target story. Rows are not semantic pairs.
for generator, sample in prepared.samples.items():
    print(generator, {"real_stories": len(sample["real"]), "synthetic_stories": len(sample["synthetic"])})


## Descriptive corpus comparisons

Length, lexical diversity, all-cross-document Jaccard and real-reference TF-IDF cosine are reported separately per generator. Exact definitions and out-of-vocabulary rates are saved. These are full-corpus descriptions; linguistic and developmental validity require the separate domain-specific analysis.


In [ ]:
from icl_pilot.evaluation.framework_descriptive import run_descriptive
descriptive_distributions, lexical_overlap = run_descriptive(prepared)
display(descriptive_distributions)
display(lexical_overlap)


## Downstream utility: all real children, shared folds, separate generators

All 238 eligible ENNI children participate: 190 TD and 48 SLI-labelled children. Five age-stratified classifier folds are independent of the 18 generation batches. Prompt-pair members stay together; unmatched TD children are single-child groups. Each fold is used once for testing, the next fold for real validation, and the other three folds for training. Thus each child is tested and validated once, with no child/prompt pair crossing partitions within a fit. There is one ordinary ENNI test set per fold, not an additional expanded test experiment.

Only synthetic bundles whose TD and SLI prompt ancestors are both in real training are eligible. Within each age/group, draw complete story bundles with replacement to obtain exactly the real training counts multiplied by the requested synthetic:real ratio. This preserves the real fold's TD/SLI and age/story proportions in real-only, synthetic-only and augmented conditions. The matched synthetic source pool is not itself treated as population prevalence. Sampler seeds and source bundle draws are shared across generators; lower ratios use nested draw prefixes. Repeated draws are logged separately from unique narratives and source children.

The primary ratio is configured in `protocol.json` (default 10:1 sampled synthetic stories per real training story). Use unweighted cross-entropy and the fixed SLI threshold. Epoch-derived update budgets and an update-matched real-only control remain explicit. BERT seeds are set before loading the random head. Validation monitors training; fixed final budgets determine checkpoints. Check synthetic BERT token lengths before fitting. Gillam is unchanged and never used for sampling or model selection.


In [ ]:
from icl_pilot.evaluation.framework_classifier import write_training_plan
CLASSIFICATION_DIR = RESULTS_DIR / "classification"
plan = pd.DataFrame([job for job, *_ in write_training_plan(prepared, CLASSIFICATION_DIR)])
print("Planned classifier fits:", len(plan))
display(plan[["training_round", "classifier_seed", "generator_key", "training_setting", "n_train", "n_real_train", "n_synthetic_train", "n_unique_synthetic_train", "effective_synthetic_to_real_ratio", "max_steps"]])


In [ ]:
from icl_pilot.evaluation.framework_classifier import run_classifiers
if RUN_CLASSIFICATION:
    story_predictions = run_classifiers(prepared, CLASSIFICATION_DIR)
else:
    story_predictions = pd.DataFrame()
    print("Classifier fitting disabled. Input checks and the explicit training plan are available.")


## Story- and child-level results, age recall and paired uncertainty

Both ENNI and Gillam receive one prediction per real story and one per child after equal-story-weight averaging. Age is used for reporting, never as a BERT input. Each ENNI child has one test fold; classifier seeds provide repeated predictions. Only observed ages are reported.

Means/SDs describe fold/seed variability. Conditional bootstrap intervals resample ENNI prompt pairs jointly and unmatched TD children individually in separate age-by-unit-type strata; Gillam children are resampled within age/group. Repeated predictions of a unit stay together. This uncertainty bootstrap is separate from the synthetic training bootstrap and does not refit models. Intervals are descriptive and unadjusted for multiplicity; they do not establish equivalence.


In [ ]:
from icl_pilot.evaluation.framework_metrics import summarize_predictions
if not story_predictions.empty:
    classification_results = summarize_predictions(story_predictions, protocol, CLASSIFICATION_DIR / "summary")
    display(classification_results["summary"])
    contrasts = classification_results["paired_differences"]
    display(contrasts.loc[contrasts["unit"].eq("child") & contrasts["metric"].isin(["balanced_accuracy", "auc"])])
    display(classification_results["age_class_recall"])
else:
    print("No scientific classifier results have been calculated in this run.")


## Separate experiment: synthetic-to-real training ratios

This optional block compares **1:1 and 10:1 sampled synthetic:real augmentation**, with one shared real-only baseline and one shared update-matched real-only control at each ratio. Use all three generators separately and the same real folds/classifier seeds. Every ratio matches the real training fold's age-by-clinical-group proportions; bootstrap draws are nested across ratios and shared across generators. Real validation and both real test datasets remain fixed. Ratios are prespecified, with no best-ratio selection on ENNI test or Gillam.

Run setup, installation/import checks and input preparation first (through `prepared = prepare_evaluation(...)`). Main classifier/descriptive/text-quality cells are not prerequisites. Set `RUN_RATIO_EXPERIMENT = True` below to fit; the default writes plans only. The five-fold, three-seed, three-generator design plans **135 fits**. Identical verified main-experiment fits are reused when available, leaving **60 additional fits** for 1:1. Synthetic-only replacement remains in the main experiment.

Outputs under `RESULTS_DIR / "ratio_experiment"` are included in the normal Drive/ZIP export. Ratios count sampled training examples, including repetitions, rather than unique generated stories. Inspect the unique counts and check synthetic BERT token lengths before enabling fitting.


In [ ]:
# Independent opt-in controls; RUN_CLASSIFICATION does not enable this block.
from icl_pilot.evaluation.framework_ratios import (
    prepare_ratio_experiment, run_ratio_experiment, summarize_ratio_experiment,
)

RUN_RATIO_EXPERIMENT = False
SYNTHETIC_REAL_RATIOS = (1, 10)  # Optional intermediate sampled ratio: (1, 5, 10).
RATIO_DIR = RESULTS_DIR / "ratio_experiment"
RATIO_REUSE_DIRS = (RESULTS_DIR / "classification",)

ratio_experiment = prepare_ratio_experiment(prepared, RATIO_DIR, ratios=SYNTHETIC_REAL_RATIOS)
ratio_plan = ratio_experiment.plan
print("Planned ratio-experiment fits (including potentially reusable jobs):", len(ratio_plan))
# ratio_level identifies the augmentation level/control comparison. For a
# real-only control, effective_synthetic_to_real_ratio remains zero.
display(ratio_plan[["ratio_level", "training_round", "classifier_seed", "generator_key",
                    "training_setting", "n_real_train", "n_synthetic_train",
                    "n_unique_synthetic_train", "effective_synthetic_to_real_ratio", "max_steps"]])


In [ ]:
if RUN_RATIO_EXPERIMENT:
    ratio_story_predictions = run_ratio_experiment(
        ratio_experiment, reuse_from=RATIO_REUSE_DIRS,
    )
else:
    ratio_story_predictions = pd.DataFrame()
    print("Ratio fitting disabled. Set RUN_RATIO_EXPERIMENT=True and rerun this cell to execute the saved plan.")


In [ ]:
if not ratio_story_predictions.empty:
    ratio_results = summarize_ratio_experiment(
        ratio_story_predictions, protocol, RATIO_DIR / "summary",
    )
    display(ratio_results["summary"].loc[lambda x: x["unit"].eq("child")])
    display(ratio_results["paired_differences"].loc[lambda x:
        x["unit"].eq("child") & x["metric"].isin(["balanced_accuracy", "auc"])])
    ratio_differences = ratio_results["ratio_differences"]
    if not ratio_differences.empty:
        display(ratio_differences.loc[lambda x:
            x["unit"].eq("child") & x["metric"].isin(["balanced_accuracy", "auc"])])

    # Means across the shared rounds/seeds; paired conditional uncertainty is
    # reported in the tables above, not inferred from this curve.
    import matplotlib.pyplot as plt
    child_summary = ratio_results["summary"].loc[lambda x: x["unit"].eq("child")]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for axis, dataset in zip(axes, ("enni", "gillam")):
        scope = child_summary.loc[child_summary["evaluation_dataset"].eq(dataset)]
        real_score = scope.loc[scope["training_setting"].eq("real-only"), "balanced_accuracy_mean"].iloc[0]
        axis.axhline(real_score, color="black", linestyle="--", alpha=.5, label="Real only")
        for generator in protocol["generator_keys"]:
            points = scope.loc[scope["generator_key"].eq(generator) &
                               scope["training_setting"].eq("synthetic-train-augment")].sort_values("ratio_level")
            axis.plot([0] + points["ratio_level"].tolist(),
                      [real_score] + points["balanced_accuracy_mean"].tolist(), marker="o", label=generator)
        axis.set(title=dataset.upper(), xlabel="Sampled synthetic stories per real training story", ylim=(0, 1))
        axis.set_xticks([0] + sorted(SYNTHETIC_REAL_RATIOS))
    axes[0].set_ylabel("Mean child-level balanced accuracy")
    axes[1].legend(fontsize="small")
    fig.tight_layout()
    fig.savefig(RATIO_DIR / "child_balanced_accuracy_by_ratio.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("No ratio-experiment classifier results have been calculated in this run.")


## Automated text quality

Pinned GPT-2 perplexity and Gaussian Fréchet distance in pinned MiniLM embeddings are computed separately per generator. Fréchet distance uses the Gaussian mean/covariance formula. Encoder limits, truncation and embedding arrays are recorded. These descriptive metrics do not establish clinical validity.


In [ ]:
from icl_pilot.evaluation.framework_quality import run_text_quality
if RUN_TEXT_QUALITY:
    text_quality_results = run_text_quality(prepared)
    display(text_quality_results)
else:
    print("Text-quality model inference disabled.")


## Save results to Drive, create a verified ZIP, and download locally

The export is **compact by default**: trained BERT `model/` and trainer/tester directories are omitted because they can make the archive several gigabytes. Set `EXPORT_INCLUDE_MODEL_CHECKPOINTS = True` in the configuration cell if checkpoint weights must be archived too.

The first cell below writes a run/environment manifest, copies the result bundle to Google Drive, creates a ZIP, checks the ZIP integrity and SHA-256 checksum, copies the ZIP to Drive, and starts the browser download. The browser's local Downloads folder is not visible to the Colab Python runtime, so local completion cannot be programmatically verified from the server.

**Only after the ZIP is visibly present on your computer should you run the final cell.** It asks for an explicit confirmation string before calling `runtime.unassign()`.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import zipfile

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

export_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
export_name = f"{EXPORT_ARCHIVE_PREFIX}_{export_timestamp}"
DRIVE_RUN_DIR = DRIVE_EXPORT_ROOT / export_name
LOCAL_EXPORT_DIR = Path("/content") / export_name
LOCAL_ZIP_PATH = Path("/content") / f"{export_name}.zip"
DRIVE_ZIP_PATH = DRIVE_EXPORT_ROOT / LOCAL_ZIP_PATH.name

def git_head(repo_dir):
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
        ).strip()
    except Exception:
        return None

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def sha256_path(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

run_manifest = {
    "export_timestamp_utc": export_timestamp,
    "evaluation_run_id": RESULTS_DIR.name,
    "project_root": str(PROJECT_ROOT),
    "results_dir": str(RESULTS_DIR),
    "icl_pilot_git_head": git_head(PROJECT_ROOT),
    "synthtexteval_git_head": git_head(SYNTHTEXTEVAL_DIR),
    "synthtexteval_requested_revision": SYNTHTEXTEVAL_REVISION,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        name: package_version(name)
        for name in [
            "SynthTextEval", "transformers", "torch", "datasets",
            "scikit-learn", "pandas", "numpy", "sentence-transformers", "evaluate", "huggingface-hub"
        ]
    },
    "configuration": {
        "protocol": protocol,
        "run_classification": RUN_CLASSIFICATION,
        "run_text_quality": RUN_TEXT_QUALITY,
        "ratio_experiment_enabled": globals().get("RUN_RATIO_EXPERIMENT", False),
        "synthetic_real_ratios": globals().get("SYNTHETIC_REAL_RATIOS"),
        "export_include_model_checkpoints": EXPORT_INCLUDE_MODEL_CHECKPOINTS,
    },
    "implementation_hashes": {
        source.name: sha256_path(source)
        for source in (PROJECT_ROOT / "src/icl_pilot/evaluation").glob("framework_*.py")
    },
    "input_files": {
        str(TEXT_RECORDS_CSV): {
            "sha256": sha256_path(TEXT_RECORDS_CSV),
            "size_bytes": TEXT_RECORDS_CSV.stat().st_size,
        },
        str(GILLAM_EXTERNAL_TEXT_CSV): {
            "sha256": sha256_path(GILLAM_EXTERNAL_TEXT_CSV),
            "size_bytes": GILLAM_EXTERNAL_TEXT_CSV.stat().st_size,
        },
    },
}
(RESULTS_DIR / "run_environment_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, default=str) + "\n"
)

# Build a clean local export tree. The original RESULTS_DIR remains untouched.
if LOCAL_EXPORT_DIR.exists():
    shutil.rmtree(LOCAL_EXPORT_DIR)

excluded_dir_names = {"__pycache__"}
if not EXPORT_INCLUDE_MODEL_CHECKPOINTS:
    excluded_dir_names.update({"model", "trainer", "tester", "external_gillam_tester"})

def export_ignore(directory, names):
    ignored = []
    for name in names:
        path = Path(directory) / name
        if path.is_dir() and name in excluded_dir_names:
            ignored.append(name)
    return ignored

shutil.copytree(RESULTS_DIR, LOCAL_EXPORT_DIR / "results", ignore=export_ignore)

# Add a checksum manifest for every portable result file.
exported_files = []
for path in sorted((LOCAL_EXPORT_DIR / "results").rglob("*")):
    if path.is_file():
        exported_files.append({
            "path": str(path.relative_to(LOCAL_EXPORT_DIR)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_path(path),
        })
(LOCAL_EXPORT_DIR / "portable_export_file_manifest.json").write_text(
    json.dumps(exported_files, indent=2) + "\n"
)

# Copy the portable directory to Drive and verify it exists before zipping.
if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(LOCAL_EXPORT_DIR, DRIVE_RUN_DIR)
assert (DRIVE_RUN_DIR / "results" / "run_environment_manifest.json").exists()

# Create the local ZIP from the portable export directory.
if LOCAL_ZIP_PATH.exists():
    LOCAL_ZIP_PATH.unlink()
shutil.make_archive(
    str(LOCAL_ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=LOCAL_EXPORT_DIR.parent,
    base_dir=LOCAL_EXPORT_DIR.name,
)
assert LOCAL_ZIP_PATH.exists() and LOCAL_ZIP_PATH.stat().st_size > 0

with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    assert bad_member is None, f"ZIP integrity check failed at: {bad_member}"
    zip_member_count = len(archive.infolist())

local_zip_sha256 = sha256_path(LOCAL_ZIP_PATH)
shutil.copy2(LOCAL_ZIP_PATH, DRIVE_ZIP_PATH)
assert DRIVE_ZIP_PATH.exists()
drive_zip_sha256 = sha256_path(DRIVE_ZIP_PATH)
assert local_zip_sha256 == drive_zip_sha256, "Drive ZIP checksum differs from local ZIP."

download_state = {
    "drive_result_directory": str(DRIVE_RUN_DIR),
    "drive_zip": str(DRIVE_ZIP_PATH),
    "local_zip_for_browser_download": str(LOCAL_ZIP_PATH),
    "zip_size_bytes": int(LOCAL_ZIP_PATH.stat().st_size),
    "zip_members": int(zip_member_count),
    "zip_sha256": local_zip_sha256,
    "server_side_checks_passed": True,
    "browser_download_confirmed": False,
}
(DRIVE_RUN_DIR / "download_state_before_local_confirmation.json").write_text(
    json.dumps(download_state, indent=2) + "\n"
)

print(json.dumps(download_state, indent=2))
print("\nServer-side Drive copy, ZIP integrity, and checksum checks passed.")
print("Starting browser download now. Do NOT run the final cell until the ZIP is visibly present on your computer.")
files.download(str(LOCAL_ZIP_PATH))


### Final step: confirm the local download, then release the Colab runtime

Run this cell **manually after checking your computer's Downloads folder**. Typing anything other than the exact confirmation string leaves the runtime assigned.

In [ ]:
# This MUST remain the final executable cell in the notebook.
confirmation = input(
    f"Confirm that {LOCAL_ZIP_PATH.name} is fully downloaded on your computer. "
    "Type exactly DOWNLOADED to unassign this Colab runtime: "
).strip()

if confirmation != "DOWNLOADED":
    raise RuntimeError(
        "Local download was not explicitly confirmed. Runtime remains assigned. "
        "Verify the ZIP in your Downloads folder, then rerun this final cell."
    )

# Recheck the durable Drive copy immediately before releasing compute.
assert DRIVE_ZIP_PATH.exists(), f"Drive ZIP is missing: {DRIVE_ZIP_PATH}"
assert sha256_path(DRIVE_ZIP_PATH) == local_zip_sha256, "Drive ZIP checksum changed."

confirmed_state = dict(download_state)
confirmed_state["browser_download_confirmed"] = True
confirmed_state["confirmation_method"] = "manual user verification in browser/local Downloads folder"
(DRIVE_RUN_DIR / "download_state_confirmed.json").write_text(
    json.dumps(confirmed_state, indent=2) + "\n"
)
print("Local download confirmed and Drive archive rechecked. Unassigning Colab runtime.")

from google.colab import runtime
runtime.unassign()